# Asset Health Metric Calculations — Interactive Demo

This notebook demonstrates the five sub-metric calculations and composite health score
used in the Asset Explorer platform.

**Standards references:**
- IEC 60076-7: Loading guide for oil-immersed transformers (FAA, hotspot model)
- IEEE C57.91: Guide for loading of mineral oil transformers (hotspot)
- IEC 60567: Oil-filled electrical equipment — DGA sampling

All data in this notebook is **synthetic mock data** — no real asset information is included.

---

In [ ]:
import sys
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Add pipeline directory to path
sys.path.insert(0, '../pipeline')
from health_metrics import (
    faa_factor, normalise_faa,
    hotspot_temperature, normalise_hotspot,
    overload_ratio, normalise_overload,
    normalise_tap, normalise_sensitivity,
    calculate_metrics_for_asset,
    TPOSC_WARNING, TPOSC_CRITICAL
)
from composite_score import composite_health_score, WEIGHTS

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.facecolor'] = '#1e2535'
plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#718096'
plt.rcParams['xtick.color'] = '#718096'
plt.rcParams['ytick.color'] = '#718096'
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.spines.top'] = False
print('Setup complete.')

## 1. FAA — Thermal Aging Acceleration Factor (IEC 60076-7)

$$
\text{FAA} = \exp\left(\frac{15000}{383} - \frac{15000}{273 + \theta_H}\right)
$$

- FAA = 1.0 at reference hotspot 110°C (383K)
- FAA = 3.6 at 120°C — aging 3.6× faster than reference
- FAA = 12.5 at 130°C — catastrophic aging rate

The reference temperature 110°C is defined in IEC 60076-7 §7.2 as the rated hotspot
for class A insulation (thermally upgraded paper) under normal loading.

In [ ]:
temps = np.linspace(60, 150, 200)
faa_values = [faa_factor(t) for t in temps]

fig, (ax1, ax2) = plt.subplots(1, 2)

# Linear scale
ax1.plot(temps, faa_values, color='#60a5fa', lw=2)
ax1.axhline(1.0,  color='#22c55e', ls='--', label='FAA=1 (reference, 110°C)')
ax1.axhline(3.6,  color='#f59e0b', ls='--', label='FAA=3.6 (120°C, Warning)')
ax1.axhline(12.5, color='#ef4444', ls='--', label='FAA=12.5 (130°C, Critical)')
ax1.set_xlabel('Hotspot Temperature (°C)')
ax1.set_ylabel('FAA (Ageing Rate)')
ax1.set_title('FAA vs Hotspot Temperature (linear)')
ax1.legend(fontsize=9)

# Log scale — shows the exponential nature more clearly
ax2.semilogy(temps, faa_values, color='#60a5fa', lw=2)
ax2.axhline(1.0,  color='#22c55e', ls='--')
ax2.axhline(3.6,  color='#f59e0b', ls='--')
ax2.axhline(12.5, color='#ef4444', ls='--')
ax2.set_xlabel('Hotspot Temperature (°C)')
ax2.set_ylabel('FAA (log scale)')
ax2.set_title('FAA vs Hotspot Temperature (log scale)')

plt.tight_layout()
plt.savefig('faa_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'FAA at 110°C: {faa_factor(110):.3f}  (should be 1.0)')
print(f'FAA at 120°C: {faa_factor(120):.2f}x reference aging rate')
print(f'FAA at 130°C: {faa_factor(130):.1f}x reference aging rate')

## 2. Hotspot Temperature Model (IEEE C57.91)

$$
\theta_H = \theta_A + \Delta\theta_{TO,R} \cdot K^{2N} + \Delta\theta_{H,R} \cdot K^{2M}
$$

Where:
- $K$ = per-unit load (MVA_actual / MVA_rated)
- $N$ = 0.9 (ONAN cooling, top-oil thermal exponent)
- $M$ = 0.8 (hotspot-to-top-oil thermal exponent)
- $\Delta\theta_{TO,R}$ = 55°C (rated top-oil rise for ONAN)
- $\Delta\theta_{H,R}$ = 30°C (rated hotspot-over-top-oil)

In [ ]:
load_ratios = np.linspace(0.5, 1.5, 100)
ambient = 25.0  # °C
top_oil_rise = 55.0  # °C

hotspots = [hotspot_temperature(ambient, top_oil_rise, k) for k in load_ratios]

plt.figure()
plt.plot(load_ratios, hotspots, color='#f59e0b', lw=2)
plt.axhline(98, color='#718096', ls=':', label='IEC reference hotspot 98°C (rated)')
plt.axhline(110, color='#f59e0b', ls='--', label='Warning: 110°C (FAA=1, limit)')
plt.axhline(120, color='#ef4444', ls='--', label='Critical: 120°C (FAA=3.6)')
plt.axvline(1.0, color='#22c55e', ls='--', alpha=0.5, label='Rated load (K=1)')
plt.fill_between(load_ratios, hotspots, 120, where=[h > 120 for h in hotspots],
                 color='#ef4444', alpha=0.2, label='Critical zone')
plt.xlabel('Load Ratio K (per-unit)')
plt.ylabel('Winding Hotspot Temperature (°C)')
plt.title(f'IEEE C57.91 Hotspot Model (Ambient={ambient}°C, ONAN)')
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()
print(f'Hotspot at K=1.0 (rated): {hotspot_temperature(ambient, top_oil_rise, 1.0):.1f}°C')
print(f'Hotspot at K=1.2 (overloaded): {hotspot_temperature(ambient, top_oil_rise, 1.2):.1f}°C')
print(f'Hotspot at K=1.5 (severely overloaded): {hotspot_temperature(ambient, top_oil_rise, 1.5):.1f}°C')

## 3. Composite Score — Fleet Snapshot

Calculate scores for the 15 transformers in our mock dataset.

In [ ]:
df = pd.read_csv('../sample_data/health_scores.csv')
print(f'Loaded {len(df)} transformers')
print(df[['asset_id','asset_name','health_score','risk_band','faa_score','hotspot_temp','overload_ratio']].to_string(index=False))

In [ ]:
# Score distribution by risk band
band_colors = {'RED': '#ef4444', 'AMBER': '#f59e0b', 'GREEN': '#22c55e'}
colors = df['risk_band'].map(band_colors)

fig, (ax1, ax2) = plt.subplots(1, 2)

# Bar chart: scores per asset
bars = ax1.barh(df['asset_id'], df['health_score'], color=colors)
ax1.axvline(70, color='#22c55e', ls='--', alpha=0.7, label='GREEN threshold (70)')
ax1.axvline(40, color='#ef4444', ls='--', alpha=0.7, label='RED threshold (40)')
ax1.set_xlabel('Health Score (0–100)')
ax1.set_title('Fleet Health Scores')
ax1.legend(fontsize=8)
ax1.invert_yaxis()

# Pie chart: band distribution
band_counts = df['risk_band'].value_counts()
ax2.pie(
    [band_counts.get(b, 0) for b in ['GREEN', 'AMBER', 'RED']],
    labels=['GREEN', 'AMBER', 'RED'],
    colors=['#22c55e', '#f59e0b', '#ef4444'],
    autopct='%1.0f%%',
    textprops={'color': '#e2e8f0'}
)
ax2.set_title('Risk Band Distribution')

plt.tight_layout()
plt.show()
print('Band counts:', band_counts.to_dict())

## 4. Composite Score Weight Decomposition

Show how each sub-metric contributes to the final score for the most critical transformer (TXF-001).

In [ ]:
# TXF-001 — most critical asset in the fleet
row = df[df['asset_id'] == 'TXF-001'].iloc[0]

metrics = calculate_metrics_for_asset(
    asset_id=row['asset_id'],
    load_mva=row['overload_ratio'] * row['capacity_mva'],
    rated_mva=row['capacity_mva'],
    ambient_temp_c=18.2,
    top_oil_rise_c=55.0,
    tap_ops=row['tap_ops_daily'],
    winding_resistance_delta_pct=row['winding_resistance_delta'],
)
score = composite_health_score(metrics)

# Sub-metric contribution bars
labels = ['FAA (30%)', 'Hotspot (25%)', 'Overload (20%)', 'Winding (10%)', 'Tap (10%)', 'Sensitivity (5%)']
max_possible = [30, 25, 20, 10, 10, 5]
actual = [
    score.faa_contribution, score.hotspot_contribution, score.overload_contribution,
    score.winding_contribution, score.tap_contribution, score.sensitivity_contribution
]
lost = [m - a for m, a in zip(max_possible, actual)]

fig, ax = plt.subplots(figsize=(10, 4))
bars_ok   = ax.barh(labels, actual, color='#22c55e', alpha=0.8, label='Score earned')
bars_lost = ax.barh(labels, lost,   left=actual, color='#ef4444', alpha=0.5, label='Score lost (degradation)')
ax.axvline(0, color='#718096', lw=0.5)
ax.set_xlabel('Score contribution (points)')
ax.set_title(f'TXF-001 — Score decomposition (Total: {score.composite_score}/100 — {score.risk_band})')
ax.legend(fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f'\n=== TXF-001 Northgate 132kV T1 ===')
print(f'Composite Score: {score.composite_score}/100  [{score.risk_band}]')
print(f'  FAA:         {score.faa_contribution:.1f}/30  (FAA={metrics.faa_factor:.2f}×, hotspot={metrics.hotspot_temp:.1f}°C)')
print(f'  Hotspot:     {score.hotspot_contribution:.1f}/25')
print(f'  Overload:    {score.overload_contribution:.1f}/20  (K={metrics.overload_ratio:.2f})')
print(f'  Winding Δ:   {score.winding_contribution:.1f}/10  ({metrics.winding_resistance_delta:.1f}%)')
print(f'  Tap ops:     {score.tap_contribution:.1f}/10  ({metrics.tap_ops_daily:.0f} ops/day)')
print(f'  Sensitivity: {score.sensitivity_contribution:.1f}/5')

## 5. FAA Sensitivity to Ambient Temperature

Shows why summer ambient temperature spikes disproportionately impact aging rate —
a 10°C rise in ambient can nearly double the effective aging rate at typical loading.

In [ ]:
load_k = 0.85  # Typical daily average loading
top_oil_rise = 55.0
ambients = [5, 15, 25, 35, 45]  # Seasonal range

fig, ax = plt.subplots()
cmap = plt.cm.get_cmap('RdYlGn_r', len(ambients))

for i, amb in enumerate(ambients):
    k_range = np.linspace(0.5, 1.3, 100)
    hs = [hotspot_temperature(amb, top_oil_rise, k) for k in k_range]
    faa_vals = [faa_factor(h) for h in hs]
    ax.plot(k_range, faa_vals, color=cmap(i), lw=2, label=f'Ambient {amb}°C')

ax.axhline(1.0, color='#718096', ls=':', alpha=0.7)
ax.axhline(3.6, color='#ef4444', ls='--', alpha=0.5, label='Warning (FAA=3.6)')
ax.axvline(1.0, color='#718096', ls=':', alpha=0.5)
ax.set_xlabel('Load Ratio K')
ax.set_ylabel('FAA (Thermal Aging Rate)')
ax.set_title('FAA Sensitivity to Ambient Temperature at Various Loads')
ax.legend(fontsize=9)
ax.set_ylim(0, 15)
plt.tight_layout()
plt.show()

print('FAA at K=0.85 (typical load) for different ambients:')
for amb in ambients:
    hs = hotspot_temperature(amb, top_oil_rise, load_k)
    faa = faa_factor(hs)
    print(f'  Ambient {amb:2d}°C → Hotspot {hs:.1f}°C → FAA {faa:.2f}x')